In [2]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:

df1 = pd.read_csv(
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    low_memory=True
)

df2 = pd.read_csv(
    "Wednesday-workingHours.pcap_ISCX.csv",
    low_memory=True
)

df3 = pd.read_csv(
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    low_memory=True
)

dataset = pd.concat(
    [df1, df2, df3],
    ignore_index=True
)

dataset.columns = dataset.columns.str.strip()

print("All 3 CSV files loaded successfully!")
print("Dataset shape:", dataset.shape)


All 3 CSV files loaded successfully!
Dataset shape: (1308978, 79)


In [4]:

# Separate input features and target labels
X = dataset.iloc[:, :-1].copy()
y = dataset.iloc[:, -1].copy()

# Convert input features to numeric values
X = X.apply(pd.to_numeric, errors="coerce")

# Replace infinite values with missing values
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Clean target labels
y = y.astype(str).str.strip()

# Remove rows with missing target labels
valid_rows = y.ne("") & y.ne("nan")
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

# Encode target labels into numbers
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(np.unique(y)))
print("Features and target prepared successfully!")


X shape: (1308978, 78)
y shape: (1308978,)
Number of classes: 11
Features and target prepared successfully!


In [5]:

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

# Fill missing values using training data only
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Train/test split completed successfully!")


Training data shape: (1047182, 78)
Testing data shape: (261796, 78)
Train/test split completed successfully!


In [5]:

from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier

algorithms = {
    "CatBoost": CatBoostClassifier(
        iterations=30,
        verbose=0,
        random_seed=0,
        thread_count=2,
        allow_writing_files=False
    ),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "SVM": SVC(kernel="rbf", random_state=0),

    "QDA": QuadraticDiscriminantAnalysis(),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=200,
        random_state=0,
        early_stopping=True
    )
}

print("Algorithms defined successfully!")
print(list(algorithms.keys()))


Algorithms defined successfully!
['CatBoost', 'KNN', 'SVM', 'QDA', 'MLP']


In [5]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting PCA=5, Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=5, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Training CatBoost...", flush=True)

    model = CatBoostClassifier(
        iterations=30,
        verbose=0,
        random_seed=0,
        thread_count=2,
        allow_writing_files=False
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test).ravel()

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(
        y_test, pred, average="weighted", zero_division=0
    ))
    print("Recall   :", recall_score(
        y_test, pred, average="weighted", zero_division=0
    ))
    print("F1 Score :", f1_score(
        y_test, pred, average="weighted", zero_division=0
    ))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nCatBoost PCA 5 experiments finished!")



Starting PCA=5, Test Size=0.2
PCA complete. Training CatBoost...
Test Size: 0.2
Accuracy : 0.983254136808813
Precision: 0.9818893566213939
Recall   : 0.983254136808813
F1 Score : 0.9825031193825425

Starting PCA=5, Test Size=0.4
PCA complete. Training CatBoost...
Test Size: 0.4
Accuracy : 0.978767819218017
Precision: 0.978392796819393
Recall   : 0.978767819218017
F1 Score : 0.9785100142157074

Starting PCA=5, Test Size=0.6
PCA complete. Training CatBoost...
Test Size: 0.6
Accuracy : 0.987834023226766
Precision: 0.9872692830489491
Recall   : 0.987834023226766
F1 Score : 0.9870239591820377

CatBoost PCA 5 experiments finished!


In [7]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting CatBoost | PCA=10 | Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=10, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Training CatBoost...", flush=True)

    model = CatBoostClassifier(
        iterations=30,
        verbose=0,
        random_seed=0,
        thread_count=2,
        allow_writing_files=False
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test).ravel()

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nCatBoost PCA 10 completed!")



Starting CatBoost | PCA=10 | Test Size=0.2
PCA complete. Training CatBoost...
Test Size: 0.2
Accuracy : 0.9585860746535471
Precision: 0.9581507536373025
Recall   : 0.9585860746535471
F1 Score : 0.9565618651047735

Starting CatBoost | PCA=10 | Test Size=0.4
PCA complete. Training CatBoost...
Test Size: 0.4
Accuracy : 0.9761340891381075
Precision: 0.9761835103066367
Recall   : 0.9761340891381075
F1 Score : 0.9755953123397131

Starting CatBoost | PCA=10 | Test Size=0.6
PCA complete. Training CatBoost...
Test Size: 0.6
Accuracy : 0.9859903461605553
Precision: 0.9850942452799566
Recall   : 0.9859903461605553
F1 Score : 0.9847938204881916

CatBoost PCA 10 completed!


In [8]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting CatBoost | PCA=15 | Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=15, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Training CatBoost...", flush=True)

    model = CatBoostClassifier(
        iterations=30,
        verbose=0,
        random_seed=0,
        thread_count=2,
        allow_writing_files=False
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test).ravel()

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nCatBoost PCA 15 completed!")



Starting CatBoost | PCA=15 | Test Size=0.2
PCA complete. Training CatBoost...
Test Size: 0.2
Accuracy : 0.9673944598084004
Precision: 0.9661396087580109
Recall   : 0.9673944598084004
F1 Score : 0.9663321854532936

Starting CatBoost | PCA=15 | Test Size=0.4
PCA complete. Training CatBoost...
Test Size: 0.4
Accuracy : 0.9630647527082156
Precision: 0.9634226378796186
Recall   : 0.9630647527082156
F1 Score : 0.9609408555913095

Starting CatBoost | PCA=15 | Test Size=0.6
PCA complete. Training CatBoost...
Test Size: 0.6
Accuracy : 0.9745615855622769
Precision: 0.9745004964122012
Recall   : 0.9745615855622769
F1 Score : 0.9735484475444997

CatBoost PCA 15 completed!


In [9]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting KNN | PCA=5 | Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=5, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Training KNN...", flush=True)

    model = KNeighborsClassifier(n_neighbors=5, n_jobs=2)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nKNN PCA 5 completed!")



Starting KNN | PCA=5 | Test Size=0.2
PCA complete. Training KNN...
Test Size: 0.2
Accuracy : 0.964602209353848
Precision: 0.9654099451316012
Recall   : 0.964602209353848
F1 Score : 0.9632866958851025

Starting KNN | PCA=5 | Test Size=0.4
PCA complete. Training KNN...
Test Size: 0.4
Accuracy : 0.9723754373634433
Precision: 0.9727556435722775
Recall   : 0.9723754373634433
F1 Score : 0.9716436694728152

Starting KNN | PCA=5 | Test Size=0.6
PCA complete. Training KNN...
Test Size: 0.6
Accuracy : 0.9959026569067224
Precision: 0.9958228816710618
Recall   : 0.9959026569067224
F1 Score : 0.9958523592293566

KNN PCA 5 completed!


In [10]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting KNN | PCA=10 | Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=10, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Training KNN...", flush=True)

    model = KNeighborsClassifier(n_neighbors=5, n_jobs=2)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nKNN PCA 10 completed!")



Starting KNN | PCA=10 | Test Size=0.2
PCA complete. Training KNN...
Test Size: 0.2
Accuracy : 0.9837774450335376
Precision: 0.9834276674821302
Recall   : 0.9837774450335376
F1 Score : 0.9827851982864617

Starting KNN | PCA=10 | Test Size=0.4
PCA complete. Training KNN...
Test Size: 0.4
Accuracy : 0.9850513376827759
Precision: 0.984762633338668
Recall   : 0.9850513376827759
F1 Score : 0.984116435590001

Starting KNN | PCA=10 | Test Size=0.6
PCA complete. Training KNN...
Test Size: 0.6
Accuracy : 0.9859406891125012
Precision: 0.9856827411180206
Recall   : 0.9859406891125012
F1 Score : 0.985016521567823

KNN PCA 10 completed!


In [11]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for test_size in [0.2, 0.4, 0.6]:

    print(f"\nStarting KNN | PCA=15 | Test Size={test_size}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=15, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print("PCA complete. Running KNN prediction...", flush=True)

    model = KNeighborsClassifier(n_neighbors=5, n_jobs=2)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"Test Size: {test_size}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nKNN PCA 15 completed!")



Starting KNN | PCA=15 | Test Size=0.2
PCA complete. Running KNN prediction...
Test Size: 0.2
Accuracy : 0.9741974667298202
Precision: 0.974671340245599
Recall   : 0.9741974667298202
F1 Score : 0.9734904126830025

Starting KNN | PCA=15 | Test Size=0.4
PCA complete. Running KNN prediction...
Test Size: 0.4
Accuracy : 0.9796711943650781
Precision: 0.9798940944461103
Recall   : 0.9796711943650781
F1 Score : 0.9792752848540408

Starting KNN | PCA=15 | Test Size=0.6
PCA complete. Running KNN prediction...
Test Size: 0.6
Accuracy : 0.9837506859675549
Precision: 0.9839906455632785
Recall   : 0.9837506859675549
F1 Score : 0.9835757617111942

KNN PCA 15 completed!


In [6]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for t in [0.2, 0.4, 0.6]:

    print(f"\nStarting SVM | PCA=5 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=t,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=5, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = LinearSVC(random_state=0, max_iter=3000, dual="auto")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"PCA = 5 | Test Size = {t}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nSVM PCA 5 completed!")



Starting SVM | PCA=5 | Test Size=0.2
PCA = 5 | Test Size = 0.2
Accuracy : 0.9088870723769653
Precision: 0.8869918669143129
Recall   : 0.9088870723769653
F1 Score : 0.8903259885334335

Starting SVM | PCA=5 | Test Size=0.4
PCA = 5 | Test Size = 0.4
Accuracy : 0.9085108252226924
Precision: 0.8865304189386183
Recall   : 0.9085108252226924
F1 Score : 0.8898979271878807

Starting SVM | PCA=5 | Test Size=0.6
PCA = 5 | Test Size = 0.6
Accuracy : 0.9084629615718111
Precision: 0.886492428240444
Recall   : 0.9084629615718111
F1 Score : 0.8898487940398233

SVM PCA 5 completed!


In [7]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# SVM - PCA 10
for t in [0.2, 0.4, 0.6]:

    print(f"\nStarting SVM | PCA=10 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=t,
        random_state=0,
        stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=10, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = LinearSVC(random_state=0, max_iter=3000, dual="auto")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"PCA = 10 | Test Size = {t}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nSVM PCA 10 completed!")



Starting SVM | PCA=10 | Test Size=0.2
PCA = 10 | Test Size = 0.2
Accuracy : 0.9140590383351923
Precision: 0.9009706945127062
Recall   : 0.9140590383351923
F1 Score : 0.8990876117385719

Starting SVM | PCA=10 | Test Size=0.4
PCA = 10 | Test Size = 0.4
Accuracy : 0.9133218230989014
Precision: 0.9001625783057485
Recall   : 0.9133218230989014
F1 Score : 0.898240835233369

Starting SVM | PCA=10 | Test Size=0.6
PCA = 10 | Test Size = 0.6
Accuracy : 0.9131663753028761
Precision: 0.8999542491746363
Recall   : 0.9131663753028761
F1 Score : 0.8980597667330468

SVM PCA 10 completed!


In [5]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for t in [0.2, 0.4, 0.6]:

    print(f"\nStarting linear SVM-style model | PCA=15 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=15, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = SGDClassifier(
        loss="hinge",
        max_iter=1000,
        tol=1e-3,
        random_state=0
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"PCA = 15 | Test Size = {t}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nLinear SVM-style PCA 15 completed!")



Starting linear SVM-style model | PCA=15 | Test Size=0.2
PCA = 15 | Test Size = 0.2
Accuracy : 0.9206481382450458
Precision: 0.909260818556996
Recall   : 0.9206481382450458
F1 Score : 0.9107306350561024

Starting linear SVM-style model | PCA=15 | Test Size=0.4
PCA = 15 | Test Size = 0.4
Accuracy : 0.9167137771394521
Precision: 0.9035294536137749
Recall   : 0.9167137771394521
F1 Score : 0.9052684764157984

Starting linear SVM-style model | PCA=15 | Test Size=0.6
PCA = 15 | Test Size = 0.6
Accuracy : 0.9144549120369958
Precision: 0.906436356256215
Recall   : 0.9144549120369958
F1 Score : 0.9040031593549882

Linear SVM-style PCA 15 completed!


In [6]:

import gc
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# QDA - PCA 5
for t in [0.2, 0.4, 0.6]:

    print(f"\nStarting QDA | PCA=5 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=5, svd_solver="randomized", random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = QuadraticDiscriminantAnalysis(reg_param=0.01)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"PCA = 5 | Test Size = {t}")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_test, pred, average="weighted", zero_division=0))
    print("F1 Score :", f1_score(y_test, pred, average="weighted", zero_division=0))

    del X_train, X_test, y_train, y_test, model, pred
    gc.collect()

print("\nQDA PCA 5 completed!")



Starting QDA | PCA=5 | Test Size=0.2
PCA = 5 | Test Size = 0.2
Accuracy : 0.660204892358936
Precision: 0.8620845498619588
Recall   : 0.660204892358936
F1 Score : 0.7182722198250384

Starting QDA | PCA=5 | Test Size=0.4
PCA = 5 | Test Size = 0.4
Accuracy : 0.6621701630277009
Precision: 0.862558823795783
Recall   : 0.6621701630277009
F1 Score : 0.7197956233521521

Starting QDA | PCA=5 | Test Size=0.6
PCA = 5 | Test Size = 0.6
Accuracy : 0.6625510735471812
Precision: 0.8627454612819042
Recall   : 0.6625510735471812
F1 Score : 0.7201465399152522

QDA PCA 5 completed!


In [17]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

qda_pca10_results = []

for t in [0.2, 0.4, 0.6]:

    print(f"\nQDA | PCA=10 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=10, random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = QuadraticDiscriminantAnalysis(
        solver="eigen",
        shrinkage=0.99
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(
        y_test, pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_test, pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_test, pred, average="weighted", zero_division=0
    )

    qda_pca10_results.append([
        "QDA", 10, t, accuracy, precision, recall, f1
    ])

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

print("\nQDA PCA 10 Final Results")
print("Algorithm | PCA | Test Size | Accuracy | Precision | Recall | F1")

for row in qda_pca10_results:
    print(row)



QDA | PCA=10 | Test Size=0.2
Accuracy : 0.6955071888034958
Precision: 0.921364886761062
Recall   : 0.6955071888034958
F1 Score : 0.7869356815534863

QDA | PCA=10 | Test Size=0.4
Accuracy : 0.6926175342633195
Precision: 0.921751270283727
Recall   : 0.6926175342633195
F1 Score : 0.7852681816481916

QDA | PCA=10 | Test Size=0.6
Accuracy : 0.6854939030057793
Precision: 0.921545488469579
Recall   : 0.6854939030057793
F1 Score : 0.780443401862954

QDA PCA 10 Final Results
Algorithm | PCA | Test Size | Accuracy | Precision | Recall | F1
['QDA', 10, 0.2, 0.6955071888034958, 0.921364886761062, 0.6955071888034958, 0.7869356815534863]
['QDA', 10, 0.4, 0.6926175342633195, 0.921751270283727, 0.6926175342633195, 0.7852681816481916]
['QDA', 10, 0.6, 0.6854939030057793, 0.921545488469579, 0.6854939030057793, 0.780443401862954]


In [18]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

qda_pca15_results = []

for t in [0.2, 0.4, 0.6]:

    print(f"\nQDA | PCA=15 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=15, random_state=0)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = QuadraticDiscriminantAnalysis(
        solver="eigen",
        shrinkage=0.99
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(
        y_test, pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_test, pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_test, pred, average="weighted", zero_division=0
    )

    qda_pca15_results.append([
        "QDA", 15, t, accuracy, precision, recall, f1
    ])

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

print("\nQDA PCA 15 Final Results")
for row in qda_pca15_results:
    print(row)



QDA | PCA=15 | Test Size=0.2
Accuracy : 0.6043369646595059
Precision: 0.8810133696139298
Recall   : 0.6043369646595059
F1 Score : 0.7113626563139106

QDA | PCA=15 | Test Size=0.4
Accuracy : 0.5991688184693426
Precision: 0.8802531387965692
Recall   : 0.5991688184693426
F1 Score : 0.7074459613920425

QDA | PCA=15 | Test Size=0.6
Accuracy : 0.6080747453166401
Precision: 0.8808394160035131
Recall   : 0.6080747453166401
F1 Score : 0.7139828892535495

QDA PCA 15 Final Results
['QDA', 15, 0.2, 0.6043369646595059, 0.8810133696139298, 0.6043369646595059, 0.7113626563139106]
['QDA', 15, 0.4, 0.5991688184693426, 0.8802531387965692, 0.5991688184693426, 0.7074459613920425]
['QDA', 15, 0.6, 0.6080747453166401, 0.8808394160035131, 0.6080747453166401, 0.7139828892535495]


In [9]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlp_pca5_results = []

for t in [0.2, 0.4, 0.6]:
    print(f"\nMLP | PCA=5 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=5, random_state=0, svd_solver="randomized")
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = MLPClassifier(
        hidden_layer_sizes=(20,),
        solver="adam",
        batch_size=4096,
        max_iter=50,
        random_state=0,
        early_stopping=True,
        n_iter_no_change=5,
        tol=0.005
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

    mlp_pca5_results.append(["MLP", 5, t, accuracy, precision, recall, f1])

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

print("\nMLP PCA 5 experiments completed!")


MLP | PCA=5 | Test Size=0.2
Accuracy : 0.9122637473452612
Precision: 0.9014554698154927
Recall   : 0.9122637473452612
F1 Score : 0.8975309804168011

MLP | PCA=5 | Test Size=0.4
Accuracy : 0.9097274213509756
Precision: 0.8930575302297418
Recall   : 0.9097274213509756
F1 Score : 0.891837778840435

MLP | PCA=5 | Test Size=0.6
Accuracy : 0.9096305388299017
Precision: 0.8878153129386118
Recall   : 0.9096305388299017
F1 Score : 0.8915252575135083

MLP PCA 5 experiments completed!


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlp_pca10_results = []

for t in [0.2, 0.4, 0.6]:
    print(f"\nMLP | PCA=10 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=10, random_state=0, svd_solver="randomized")
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = MLPClassifier(
        hidden_layer_sizes=(20,),
        solver="adam",
        batch_size=4096,
        max_iter=50,
        random_state=0,
        early_stopping=True,
        n_iter_no_change=5,
        tol=0.005
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

    mlp_pca10_results.append(
        ["MLP", 10, t, accuracy, precision, recall, f1]
    )

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

print("\nMLP PCA 10 experiments completed!")


MLP | PCA=10 | Test Size=0.2
Accuracy : 0.9537540680529878
Precision: 0.9420181522298972
Recall   : 0.9537540680529878
F1 Score : 0.9465072795741996

MLP | PCA=10 | Test Size=0.4
Accuracy : 0.9511107885529191
Precision: 0.9392952363983218
Recall   : 0.9511107885529191
F1 Score : 0.9436117688649216

MLP | PCA=10 | Test Size=0.6
Accuracy : 0.9500806608716467
Precision: 0.938588147369259
Recall   : 0.9500806608716467
F1 Score : 0.9421463790907268

MLP PCA 10 experiments completed!


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlp_pca15_results = []

for t in [0.2, 0.4, 0.6]:
    print(f"\nMLP | PCA=15 | Test Size={t}", flush=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=t, random_state=0, stratify=y
    )

    imputer = SimpleImputer(strategy="mean")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    pca = PCA(n_components=15, random_state=0, svd_solver="randomized")
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    model = MLPClassifier(
        hidden_layer_sizes=(20,),
        solver="adam",
        batch_size=4096,
        max_iter=50,
        random_state=0,
        early_stopping=True,
        n_iter_no_change=5,
        tol=0.005
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

    mlp_pca15_results.append(
        ["MLP", 15, t, accuracy, precision, recall, f1]
    )

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

print("\nMLP PCA 15 experiments completed!")


MLP | PCA=15 | Test Size=0.2
Accuracy : 0.9540558297300188
Precision: 0.9427085633687502
Recall   : 0.9540558297300188
F1 Score : 0.9470595145727629

MLP | PCA=15 | Test Size=0.4
Accuracy : 0.9526520649666153
Precision: 0.9411731461192289
Recall   : 0.9526520649666153
F1 Score : 0.9454763930768079

MLP | PCA=15 | Test Size=0.6
Accuracy : 0.9517142504268596
Precision: 0.9402803543252968
Recall   : 0.9517142504268596
F1 Score : 0.9443568030003197

MLP PCA 15 experiments completed!
